# Teach Qwen3 4B to use v's tools (free GPU)

This notebook trains a small add-on (LoRA) that makes Qwen3 4B, the model v uses on 8 GB laptops, better at v's
everyday jobs: finding and reading files, editing them, reading photos, answering from the web, clicking and
filling in forms, and taking no for an answer. Then it scores the model **before and after** with `v eval` on tasks
it never trained on, and makes a GGUF file that v runs.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (Colab), or Accelerator → **GPU T4** and Internet → **on** (Kaggle, in the notebook settings).
Everything takes about 1.5 to 2 hours on a free T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get v and the training tools

In [ ]:
!git clone --depth 1 https://github.com/psaravanansep2/v
%pip install -q -e "./v[free,local]" -r v/training/requirements.txt

## 2. Make the training examples

Known-good solutions to v's everyday tasks, played through v's real tools, written exactly the way v talks to the
model. Two kinds of task (`todo`, `open`) are held out, and the eval uses different wordings and details, so the
scores below show whether the skills carry over.

In [ ]:
!python v/training/make_data.py --per-family 120 --out data/train.jsonl

## 3. Score the model as it is now

A small server runs the model on the GPU; `v eval` gives it 3 tasks of each kind (45 in all).

In [ ]:
import subprocess, time, urllib.request

def serve(model):
    proc = subprocess.Popen(["python", "v/training/serve_hf.py", "--model", model, "--port", "8000"])
    for _ in range(600):
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2)
            return proc
        except OSError:
            time.sleep(2)
    raise RuntimeError("the model server didn't start")

server = serve("Qwen/Qwen3-4B-Instruct-2507")
!v eval --local-url http://127.0.0.1:8000 --per-family 3 --json before.json
server.terminate(); server.wait()

## 4. Train (about an hour)

In [ ]:
!python v/training/finetune.py --data data/train.jsonl --out runs/v-qwen3-4b

## 5. Score it again, on the same tasks

In [ ]:
server = serve("runs/v-qwen3-4b/merged")
!v eval --local-url http://127.0.0.1:8000 --per-family 3 --json after.json
server.terminate(); server.wait()

import json
before, after = json.load(open("before.json")), json.load(open("after.json"))
def by_kind(results):
    out = {}
    for t in results["tasks"]:
        out.setdefault(t["family"], []).append(t["passed"])
    return out
b, a = by_kind(before), by_kind(after)
print(f"{'kind of task':<22}{'before':>8}{'after':>8}")
for kind in b:
    print(f"{kind:<22}{sum(b[kind]):>5}/{len(b[kind])}{sum(a.get(kind, [])):>5}/{len(a.get(kind, []))}")
print(f"{'all':<22}{before['passed']:>5}/{before['total']}{after['passed']:>5}/{after['total']}")
print("(never trained on: 'read, then write' and 'open a file')")

## 6. Make the GGUF file for v

In [ ]:
!python v/training/export_gguf.py --model runs/v-qwen3-4b/merged --out v-qwen3-4b-Q4_K_M.gguf

## 7. Take it home

Download the file (Colab: below; Kaggle: Output panel), then on your computer:

```
v --local-model ~/Downloads/v-qwen3-4b-Q4_K_M.gguf
v eval --local-model ~/Downloads/v-qwen3-4b-Q4_K_M.gguf    # its score on your computer
```

To share it, upload it to a Hugging Face model repo; then anyone can use
`v --local-model hf:<you>/<repo>/v-qwen3-4b-Q4_K_M.gguf`.

In [ ]:
try:
    from google.colab import files
    files.download("v-qwen3-4b-Q4_K_M.gguf")
except ImportError:
    print("Not on Colab: find v-qwen3-4b-Q4_K_M.gguf in the output files.")